# SPY 0DTE Flow-Following Strategy — Clustered Mode

Scan the 0DTE option trade tape for **multi-print buy clusters**: ≥ 3 buy-aggressor prints, each ≥ `size_threshold` contracts, hitting *same-direction* (all calls OR all puts) contracts within the ATM ±$10 strike window during the same 1-minute candle. When a cluster fires we buy the dominant strike (most prints in that minute; tiebreak: closest to ATM) and manage with PT / SL. One trade per day; first qualifying cluster wins. Trades run until profit target, stop loss, or 3:55 PM ET.

**Aggressor classification** (rough proxy without NBBO quotes):
- `trade_price >= bar_open` → buy aggressor → counted toward cluster
- `trade_price <  bar_open` → sell aggressor → skipped

**Multi-leg filter:** prints whose Polygon/OPRA condition codes mark them as spread / vertical / stock-tied trades are skipped (`exclude_multi_leg=True` by default).

**Sweep grid:** 150 configs (5 size × 3 PT × 5 SL × 2 entry modes), N=3 fixed.
- size: 500, 1000, 1500, 2000, 2500 (per-print floor — real sweep legs land here)
- pt: 5%, 10%, 15%
- sl: 20%, 25%, 30%, 40%, 50%

Strike window: ±$10 of opening spot (wider than before so we catch flow further from ATM).


## 1. Setup

In [ ]:
import os, sys
REPO = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'
if not os.path.isdir(REPO):
    !git clone --quiet -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO}
else:
    !cd {REPO} && git pull --quiet
SRC = REPO + '/src'
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO)
print('OK')

## 2. Paste your Polygon key

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Smoke test — one recent day
Confirms trade-tape fetch + aggressor classification + entry pipeline. Uses single-print mode internally — just a code-path sanity check.

In [ ]:
!python -m iron_condor.cli --smoke -v

## 4. Clustered sweep — past 7 days
~1 min. Sanity-checks that some clusters fire. With only ~5 trading days the win-rate and return numbers are noise; we're checking that the pipeline produces non-zero trades.

In [ ]:
!python -m iron_condor.cli --days 7 --sweep --signal-mode clustered

## 4b. Clustered sweep — past 30 days
~5–10 min. ~21 trading days, enough to start seeing whether top configs have a believable win-rate distribution.

In [ ]:
!python -m iron_condor.cli --days 30 --sweep --signal-mode clustered

## 4c. Clustered sweep — past 60 days
~10–15 min. ~42 trading days. Bridges the gap between the 30-day "looks promising" and a full year commitment.

In [ ]:
!python -m iron_condor.cli --days 60 --sweep --signal-mode clustered

## 4d. Clustered sweep — past 365 days (optional)
Full year — only run if cell § 4c looked promising. Trade tapes are cached so re-running is fast once the first sweep has populated the cache.

In [ ]:
!python -m iron_condor.cli --days 365 --sweep --signal-mode clustered

## 4e. Diagnostic — see why N=3 fires (or doesn't)
Runs ONE clustered config (sz=1000, pt=10, sl=30) over 7 days with verbose logging. Each per-day line shows:

- **large** = total prints ≥ S across all ATM contracts that day
- **multi_leg** = filtered by OPRA condition codes
- **outside_window** = print landed before 9:35 or after 15:00 ET
- **no_bar** = no 1-min option bar at that minute (sparse pre/post-open trades)
- **unknown_agg** = couldn't classify (no bar_open available)
- **sell** = filtered as sell-aggressor
- **buy_kept** = qualifying buy-aggressor prints
- **max_per_minute** = biggest same-direction cluster in any 1-min candle (key metric)
- **clusters_found** = minutes that hit ≥ N

If `max_per_minute` is < 3, drop S further or expand window.

In [ ]:
!python -m iron_condor.cli --days 7 --sweep --signal-mode clustered --size-threshold 1000 --pt 0.10 --sl 0.30 -v 2>&1 | grep -E "(clustered S|Sweep)"

## 5. Top 20 configs
Reads `results/sweep_summary.csv` written by whichever sweep cell you last ran.

In [ ]:
import pandas as pd
summary = pd.read_csv('results/sweep_summary.csv')
summary.head(20)

## 6. Size-threshold breakdown — does bigger = better?
Per-threshold average return, win rate, trade count. Useful to see if 'bigger prints = stronger signal' actually holds.

In [ ]:
import re
rows = []
for _, r in summary.iterrows():
    m = re.match(r'sz(\d+)\|', r['config'])
    if m:
        rows.append({
            'sz': int(m.group(1)),
            'return': r['total_return_pct'],
            'win_rate': r['win_rate'],
            'trades': r['n_trades'],
        })
grid = pd.DataFrame(rows)
print('=== Avg return % by size threshold ===')
print(grid.groupby('sz')['return'].agg(['mean', 'median', 'max']).round(1))
print('\n=== Avg win rate by size threshold ===')
print(grid.groupby('sz')['win_rate'].agg(['mean', 'median', 'max']).round(2))
print('\n=== Trade count by size threshold ===')
print(grid.groupby('sz')['trades'].agg(['mean', 'median', 'min', 'max']).round(0))

## 6b. Entry-mode comparison — does instant beat next-bar?
If the cluster is genuinely informative, instant entries should win — we capture the directional move that next-bar entries miss in the first 60 sec.

In [ ]:
for em_short, label in [('inst', 'instant'), ('next', 'next_bar_open')]:
    s = summary[summary['config'].str.contains(f'em={em_short}')]
    if not s.empty:
        best = s.iloc[0]
        print(f'\n=== Best {label.upper()} config ===')
        print(f"  {best['config']}")
        print(f"  return={best['total_return_pct']:>+8.1f}%")
        print(f"  win_rate={best['win_rate']:>6.1%}")
        print(f"  trades={best['n_trades']}")
        print(f"  avg_pnl=${best['avg_net_pnl']:>+8.2f}")

## 7. Top config breakdown — call vs put, exit reasons

In [ ]:
trades = pd.read_csv('results/sweep_trades.csv')
top_cfg = summary.iloc[0]['config']
sub = trades[trades['config'] == top_cfg]
taken = sub[sub['exit_reason'].isin(['profit', 'stop', 'time_stop'])]
print(f'Top config: {top_cfg}')
print(f'Total days: {len(sub)}, taken trades: {len(taken)}')
print('\n=== Call vs put performance ===')
if not taken.empty and 'right' in taken.columns:
    print(taken.groupby('right').agg(
        trades=('net_pnl', 'count'),
        win_rate=('net_pnl', lambda s: (s > 0).mean()),
        avg_pnl=('net_pnl', 'mean'),
        total_pnl=('net_pnl', 'sum'),
    ).round(2))
print('\n=== Exit reasons ===')
print(sub['exit_reason'].value_counts())
print('\n=== Print sizes captured (taken trades only) ===')
if not taken.empty and 'print_size' in taken.columns:
    print(taken['print_size'].describe().round(0))

## 8. Equity curve — top 3 configs

In [ ]:
import matplotlib.pyplot as plt
top3 = summary.head(3)['config'].tolist()
fig, ax = plt.subplots(figsize=(11, 5))
for cfg in top3:
    sub = trades[trades['config'] == cfg].sort_values('day')
    ax.plot(pd.to_datetime(sub['day']), sub['balance_after'], label=cfg, linewidth=1.5)
ax.axhline(1500, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date'); ax.set_ylabel('Balance ($)')
ax.set_title('Clustered flow — top 3 equity curves')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Per-month breakdown — regime or fluke?
Splits the year by calendar month. Two views:
1. **Aggregate across all configs** — does any month look different (high WR across the board)?
2. **Top config monthly** — was the winner's edge spread across the year or concentrated in 1–2 months?

In [ ]:
trades_all = pd.read_csv('results/sweep_trades.csv')
trades_all['day'] = pd.to_datetime(trades_all['day'])
trades_all['month'] = trades_all['day'].dt.to_period('M').astype(str)
taken_all = trades_all[trades_all['exit_reason'].isin(['profit', 'stop', 'time_stop'])]

print('=== Per-month aggregate (all configs averaged) ===')
monthly = taken_all.groupby('month').agg(
    trades=('net_pnl', 'count'),
    win_rate=('net_pnl', lambda s: (s > 0).mean()),
    avg_pnl=('net_pnl', 'mean'),
    total_pnl=('net_pnl', 'sum'),
).round(2)
print(monthly.to_string())

print(f'\n=== Top config ({summary.iloc[0]["config"]}) per month ===')
top_cfg = summary.iloc[0]['config']
top_taken = taken_all[taken_all['config'] == top_cfg]
if not top_taken.empty:
    top_monthly = top_taken.groupby('month').agg(
        trades=('net_pnl', 'count'),
        win_rate=('net_pnl', lambda s: (s > 0).mean()),
        avg_pnl=('net_pnl', 'mean'),
        total_pnl=('net_pnl', 'sum'),
    ).round(2)
    print(top_monthly.to_string())
else:
    print('(no taken trades for top config)')